In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import json

from src.features import compute_hedge_ratio, compute_kalman_hedge, compute_calendar_features, compute_spread_vol
from src.labels import get_daily_vol, get_vertical_barrier, apply_pt_sl_on_t1, get_bins, get_avg_uniqueness, seq_bootstrap
from src.cv import temporal_split, WalkForwardPurgedCV, cv_score, feat_importance_mda, feat_importance_mdi

## 1. Features

Compute all features on the full dataset before splitting.

In [2]:
df = pd.read_csv("../data/full_dataset.csv", index_col="date", parse_dates=True)
df = compute_hedge_ratio(df, 'close_corn', 'close_soybean')         # OLS (504-day window)
df = compute_kalman_hedge(df, 'close_corn', 'close_soybean', delta=1e-6)        # Kalman 3-state
df = compute_calendar_features(df)                                   # month, day_of_week
df = compute_spread_vol(df, 'close_corn', 'close_soybean', 'kf_hedge_ratio')  # EWMA vol

Window:          504 trading days
Valid rows:      3148 / 3652
Hedge ratio:     -0.0992 to 1.3937
Intercept:       -5.8013 to 23.9495
Spread mean:     0.0989
Converged in 4 iteration(s)
  phi      = 0.994075  (half-life = 168 trading days)
  sigma2_s = 0.030082
  delta    = 1.0e-06  (hyperparameter — tune via CV)
  R        = 1.0e-06  (numerical floor)
Hedge ratio:      0.2787 to 0.6499
Spread level std: 1.5955
Innovation z std: 1.1045 (ideal ~ 1.0)
Level z std:      1.1798


## 2. Labels

Triple barrier labeling (AFML Ch.3) + sample uniqueness weights (Ch.4).
Restrict to dates where OLS hedge ratio is valid (after 504-day burn-in).

In [3]:
corn        = df['close_corn']
soy         = df['close_soybean']
hedge_ratio = df['hedge_ratio']

# Only label dates where the OLS hedge ratio exists
valid_dates = df.dropna(subset=['hedge_ratio']).index

vol     = get_daily_vol(corn, soy, hedge_ratio)
t1      = get_vertical_barrier(valid_dates, num_days=200, t_events=valid_dates)
touches = apply_pt_sl_on_t1(corn, soy, hedge_ratio, t1, vol, pt_sl=[2.5, 2.5])
labels  = get_bins(touches, corn, soy, hedge_ratio)
weights = get_avg_uniqueness(labels, df.index)

print(f"\nLabeled events: {len(labels)}")
print(f"Label distribution: {labels['bin'].value_counts().to_dict()}")
print(f"Mean uniqueness: {weights.mean():.4f}")


Labeled events: 2948
Label distribution: {1: 1527, -1: 1421}
Mean uniqueness: 0.0881


## 3. Assemble X, y, t1, w

Build the full feature matrix: 4 Kalman-derived + 2 calendar + 1 spread vol + 90 weather = 97 features.
Exclude raw prices, volumes, and intermediate estimation columns (hedge_ratio, intercept, spread)
which would leak information or are not meaningful as predictors.

In [4]:
# Columns to EXCLUDE from features
exclude_cols = [
    'close_corn', 'close_soybean', 'high_corn', 'high_soybean',
    'low_corn', 'low_soybean', 'open_corn', 'open_soybean',
    'volume_corn', 'volume_soybean',
    'hedge_ratio', 'intercept','spread',       # OLS intermediates
    'kf_hedge_ratio', 'kf_intercept',            # Kalman intermediates
]

feature_cols = [c for c in df.columns if c not in exclude_cols]

X         = df.loc[labels.index, feature_cols].copy()
y         = labels['bin'].map({-1: 0, 1: 1})   # XGBoost needs {0, 1}
t1_series = labels['t1']
w         = weights

# Sanity check for NaNs
nan_count = X.isna().sum()
if nan_count.any():
    print(f"WARNING: NaN features found:")
    print(nan_count[nan_count > 0])
else:
    print(f"No NaN features — all {X.shape[0]} rows clean")

print(f"\nFeatures: {len(feature_cols)}")
print(f"  Weather:  {len([c for c in feature_cols if 'roll30d' in c])}")
print(f"  Kalman:   {[c for c in feature_cols if c.startswith('kf_')]}")
print(f"  Other:    {[c for c in feature_cols if 'roll30d' not in c and not c.startswith('kf_')]}")
print(f"\nShape: {X.shape}")
print(f"Label dist: {y.value_counts().sort_index().to_dict()}")

No NaN features — all 2948 rows clean

Features: 97
  Weather:  90
  Kalman:   ['kf_spread', 'kf_innovation', 'kf_z_score', 'kf_level_z']
  Other:    ['month', 'day_of_week', 'spread_vol']

Shape: (2948, 97)
Label dist: {0: 1421, 1: 1527}


## 4. Dev / Holdout Split

Reserve the last 1.5 years (375 trading days) as a final holdout set.
All model development happens on the dev set only. Holdout is touched once at the end.

In [5]:
split = temporal_split(X, y, t1_series, sample_weight=w, n_holdout=375)

Dev set:      2573 obs  (2013-09-20 → 2023-12-08)
Holdout set:   375 obs  (2023-12-11 → 2025-06-10)
Cutoff date: 2023-12-11
Dev label dist:     {0: 1205, 1: 1368}
Holdout label dist: {0: 216, 1: 159}


## 5. Baseline: 6-Feature XGBoost

Kalman + calendar + spread vol only. Default hyperparameters.
Establishes the baseline to beat.

In [17]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300, # Reduced from 100
    max_depth=2,       # Reduced from 3
    learning_rate=0.1,
    min_child_weight=15, 
    eval_metric='logloss',
    random_state=42,
)

baseline_cols = ['kf_level_z', 'kf_z_score', 'kf_spread', 'spread_vol', 'month', 'day_of_week']
split_base = temporal_split(X[baseline_cols], y, t1_series, sample_weight=w, n_holdout=375)

cv_base = WalkForwardPurgedCV(n_periods=3, t1=split_base['t1_dev'])
print("\n=== Baseline (6 features) — Accuracy ===")
res_base = cv_score(model, split_base['X_dev'], split_base['y_dev'], split_base['t1_dev'],
                    sample_weight=split_base['w_dev'], cv=cv_base)

Dev set:      2573 obs  (2013-09-20 → 2023-12-08)
Holdout set:   375 obs  (2023-12-11 → 2025-06-10)
Cutoff date: 2023-12-11
Dev label dist:     {0: 1205, 1: 1368}
Holdout label dist: {0: 216, 1: 159}

=== Baseline (6 features) — Accuracy ===
Fold    Train   Test  Purged               Train Dates                Test Dates  Train Score  Test Score
---------------------------------------------------------------------------------------------------------
1         846    858      11   2013-09-20 → 2017-02-03   2017-02-15 → 2020-07-14       0.5579      0.4674
2        1686    858      29   2013-09-20 → 2020-06-30   2020-07-15 → 2023-12-08       0.5718      0.5175
---------------------------------------------------------------------------------------------------------
Mean                                                                                   0.5648      0.4924
Std                                                                                    0.0069      0.0251


## 6. Full Feature Set XGBoost (97 features)

All weather + Kalman + calendar + spread vol. Same default hyperparameters.
Expect more overfitting but potentially better test performance if weather carries signal.

In [18]:
cv_full = WalkForwardPurgedCV(n_periods=3, t1=split['t1_dev'])

print("=== Full Feature Set (97 features) — Accuracy ===")
res_full = cv_score(model, split['X_dev'], split['y_dev'], split['t1_dev'],
                    sample_weight=split['w_dev'], cv=cv_full)

print("\n=== Full Feature Set — Neg Log Loss ===")
cv_full_ll = WalkForwardPurgedCV(n_periods=3, t1=split['t1_dev'])
res_full_ll = cv_score(model, split['X_dev'], split['y_dev'], split['t1_dev'],
                       sample_weight=split['w_dev'], cv=cv_full_ll, scoring='neg_log_loss')

=== Full Feature Set (97 features) — Accuracy ===
Fold    Train   Test  Purged               Train Dates                Test Dates  Train Score  Test Score
---------------------------------------------------------------------------------------------------------
1         846    858      11   2013-09-20 → 2017-02-03   2017-02-15 → 2020-07-14       0.5579      0.4674
2        1686    858      29   2013-09-20 → 2020-06-30   2020-07-15 → 2023-12-08       0.6673      0.6084
---------------------------------------------------------------------------------------------------------
Mean                                                                                   0.6126      0.5379
Std                                                                                    0.0547      0.0705

=== Full Feature Set — Neg Log Loss ===
Fold    Train   Test  Purged               Train Dates                Test Dates  Train Score  Test Score
-------------------------------------------------------------

## 7. MDA Feature Importance (AFML Ch.8)

Permutation importance using walk-forward CV. For each feature, shuffle it and
measure the drop in OOS performance. Positive = useful, negative = detrimental.

In [19]:
cv_mda = WalkForwardPurgedCV(n_periods=3, t1=split['t1_dev'])
mda = feat_importance_mda(model, split['X_dev'], split['y_dev'], split['t1_dev'],
                          sample_weight=split['w_dev'], cv=cv_mda)

print("Top 20 features:")
print(mda.head(20).to_string())
print(f"\nBottom 10:")
print(mda.tail(10).to_string())
print(f"\nPositive importance: {(mda['mean'] > 0).sum()} / {len(mda)}")

Top 20 features:
                                                             mean       std
kf_spread                                                0.012468  0.012468
temperature_2m_max_mato_grosso_roll30d_mean              0.010553  0.010553
soil_moisture_28_to_100cm_mean_central_iowa_roll30d_min  0.010067  0.010067
precipitation_sum_central_indiana_roll30d_sum            0.009883  0.009883
wind_speed_10m_max_central_iowa_roll30d_max              0.003563  0.003563
kf_level_z                                               0.002070  0.002070
wind_speed_10m_max_parana_roll30d_max                    0.001627  0.001627
soil_moisture_7_to_28cm_mean_rio_grande_sul_roll30d_min  0.001493  0.001493
soil_moisture_0_to_7cm_mean_parana_roll30d_min           0.000662  0.000662
relative_humidity_2m_mean_central_illinois_roll30d_mean  0.000408  0.000408
soil_moisture_0_to_7cm_mean_rio_grande_sul_roll30d_min   0.000055  0.000055
soil_temperature_0_to_7cm_mean_mato_grosso_roll30d_mean  0.000000  0.00

## 8. MDI Feature Importance

In [20]:

mdi = feat_importance_mdi(split['X_dev'], split['y_dev'], sample_weight=split['w_dev'])

print("MDI Top 20:")
print(mdi.head(20).to_string())

MDI Top 20:
                                                                 mean       std
kf_spread                                                    0.015785  0.000606
kf_level_z                                                   0.015595  0.000542
spread_vol                                                   0.012916  0.000488
temperature_2m_min_mato_grosso_roll30d_mean                  0.012711  0.000464
et0_fao_evapotranspiration_sum_central_indiana_roll30d_sum   0.012318  0.000449
soil_temperature_0_to_7cm_mean_mato_grosso_roll30d_mean      0.012115  0.000416
temperature_2m_mean_mato_grosso_roll30d_mean                 0.011914  0.000419
temperature_2m_mean_rio_grande_sul_roll30d_mean              0.011887  0.000369
vapour_pressure_deficit_max_central_indiana_roll30d_mean     0.011874  0.000418
temperature_2m_max_mato_grosso_roll30d_mean                  0.011854  0.000373
precipitation_sum_central_indiana_roll30d_sum                0.011798  0.000429
soil_moisture_28_to_100cm_me

## 9. Union of MDI & MDA

In [21]:
mdi_top20 = set(mdi.head(20).index)
mda_top20 = set(mda.head(20).index)
union = mdi_top20 | mda_top20

vetoed = [f for f in union if mda.loc[f, 'mean'] < 0]
selected = sorted([f for f in union if mda.loc[f, 'mean'] >= 0])


print(f"MDI top 20:        {len(mdi_top20)}")
print(f"MDA top 20:        {len(mda_top20)}")
print(f"Union:             {len(union)}")
print(f"Vetoed (MDA < 0):  {len(vetoed)}")
print(f"Final selection:   {len(selected)}")

print(f"\nVetoed features:")
for f in vetoed:
    print(f"  {f}: MDI rank {list(mdi.index).index(f)+1}, MDA = {mda.loc[f,'mean']:.4f}")

print(f"\nSelected features ({len(selected)}):")
print(f"{'Feature':<65} {'MDI rank':>9} {'MDA rank':>9} {'MDA mean':>9}")
print("-" * 95)
for f in selected:
    mdi_rank = list(mdi.index).index(f) + 1
    mda_rank = list(mda.index).index(f) + 1
    print(f"{f:<65} {mdi_rank:>9} {mda_rank:>9} {mda.loc[f,'mean']:>9.4f}")

with open('../data/selected_weather_features.json', 'w') as f:
    json.dump([c for c in selected if c not in ['kf_spread', 'kf_innovation']], f)

MDI top 20:        20
MDA top 20:        20
Union:             33
Vetoed (MDA < 0):  1
Final selection:   32

Vetoed features:
  temperature_2m_min_mato_grosso_roll30d_mean: MDI rank 4, MDA = -0.0014

Selected features (32):
Feature                                                            MDI rank  MDA rank  MDA mean
-----------------------------------------------------------------------------------------------
et0_fao_evapotranspiration_sum_central_indiana_roll30d_sum                5        81    0.0000
et0_fao_evapotranspiration_sum_central_iowa_roll30d_sum                  16        64    0.0000
et0_fao_evapotranspiration_sum_mato_grosso_roll30d_sum                   14        19    0.0000
et0_fao_evapotranspiration_sum_parana_roll30d_sum                        48        13    0.0000
kf_level_z                                                                2         6    0.0021
kf_spread                                                                 1         1    0.0125
precipi

## 10. Retrain XGBoost with Selected Features


In [22]:
# Cell 10: Retrain with selected features (uses `selected` from cell 9)
print(f"Selected features: {len(selected)}\n")

split_sel = temporal_split(X[selected], y, t1_series, sample_weight=w, n_holdout=375)

cv_sel = WalkForwardPurgedCV(n_periods=3, t1=split_sel['t1_dev'])
print("\n=== XGBoost with selected features ===")
res_sel = cv_score(model, split_sel['X_dev'], split_sel['y_dev'], split_sel['t1_dev'],
                   sample_weight=split_sel['w_dev'], cv=cv_sel)

Selected features: 32

Dev set:      2573 obs  (2013-09-20 → 2023-12-08)
Holdout set:   375 obs  (2023-12-11 → 2025-06-10)
Cutoff date: 2023-12-11
Dev label dist:     {0: 1205, 1: 1368}
Holdout label dist: {0: 216, 1: 159}

=== XGBoost with selected features ===
Fold    Train   Test  Purged               Train Dates                Test Dates  Train Score  Test Score
---------------------------------------------------------------------------------------------------------
1         846    858      11   2013-09-20 → 2017-02-03   2017-02-15 → 2020-07-14       0.5579      0.4674
2        1686    858      29   2013-09-20 → 2020-06-30   2020-07-15 → 2023-12-08       0.6435      0.6329
---------------------------------------------------------------------------------------------------------
Mean                                                                                   0.6007      0.5501
Std                                                                                    0.0428      0.

## 11. Random Forest with Sequential Bootstrap (AFML Ch.4.5)

RF requires sequential bootstrap because standard bootstrap oversamples redundant
observations (mean uniqueness ≈ 0.12). We draw bags proportional to each observation's
uniqueness, then train individual decision trees and average predictions.

In [15]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, log_loss
from sklearn.base import clone

n_trees = 200

# Evaluate on each walk-forward fold
cv_rf = WalkForwardPurgedCV(n_periods=3, t1=split_sel['t1_dev'])

for fold_i, (train_idx, test_idx) in enumerate(cv_rf.split(split_sel['X_dev'])):
    X_train = split_sel['X_dev'].iloc[train_idx]
    y_train = split_sel['y_dev'].iloc[train_idx]
    w_train = split_sel['w_dev'].iloc[train_idx]
    X_test  = split_sel['X_dev'].iloc[test_idx]
    y_test  = split_sel['y_dev'].iloc[test_idx]
    
    t1_train = split_sel['t1_dev'].iloc[train_idx]
    
    avg_u = w_train.mean()
    s_length = max(int(avg_u * len(t1_train)), 50)
    
    print(f"Fold {fold_i+1}: train={len(X_train)}, s_length={s_length}")
    
    # Train trees on sequential bootstrap bags
    trees = []
    for tree_i in range(n_trees):
        bag_idx = seq_bootstrap(t1_train, df.index, s_length=s_length,
                               random_state=tree_i)
        
        dt = DecisionTreeClassifier(
            criterion='entropy',
            max_features='sqrt',
            max_depth=5,
            min_samples_leaf=max(1, int(0.05 * s_length)),
            class_weight='balanced',
            random_state=tree_i,
        )
        dt.fit(X_train.iloc[bag_idx].values, y_train.iloc[bag_idx].values,
               sample_weight=w_train.iloc[bag_idx].values)
        trees.append(dt)
    
    # Predict: average probabilities
    probs = np.array([t.predict_proba(X_test.values) for t in trees])
    avg_prob = probs.mean(axis=0)
    pred = (avg_prob[:, 1] >= 0.5).astype(int)
    
    acc = accuracy_score(y_test.values, pred)
    ll = log_loss(y_test.values, avg_prob)
    print(f"  Seq Bootstrap RF:  acc={acc:.4f}, neg_ll={-ll:.4f}")
    
    # Compare with XGBoost on same fold
    m = clone(model)
    m.fit(X_train.values, y_train.values, sample_weight=w_train.values)
    xgb_acc = accuracy_score(y_test.values, m.predict(X_test.values))
    xgb_ll = log_loss(y_test.values, m.predict_proba(X_test.values))
    print(f"  XGBoost:           acc={xgb_acc:.4f}, neg_ll={-xgb_ll:.4f}\n")

Fold 1: train=846, s_length=69
  Seq Bootstrap RF:  acc=0.4091, neg_ll=-0.7875
  XGBoost:           acc=0.4674, neg_ll=-0.6987

Fold 2: train=1686, s_length=131
  Seq Bootstrap RF:  acc=0.5956, neg_ll=-0.6682
  XGBoost:           acc=0.6515, neg_ll=-0.6549



## 12. XGBoost Hyperparameter Tuning

In [16]:
from itertools import product
from sklearn.base import clone
from sklearn.metrics import accuracy_score, log_loss

param_grid = list(product(
    [2, 3, 4, 5, 6],           # max_depth
    [0.01, 0.05, 0.1],   # learning_rate
    [50, 75, 100, 200, 300],      # n_estimators
    [1, 5, 10, 15, 20, 25, 30],          # min_child_weight
))

print(f"Testing {len(param_grid)} XGBoost configurations...\n")

xgb_results = []
for md, lr, ne, mcw in param_grid:
    m = XGBClassifier(max_depth=md, learning_rate=lr, n_estimators=ne,
                      min_child_weight=mcw, eval_metric='logloss', random_state=42)
    
    cv = WalkForwardPurgedCV(n_periods=3, t1=split_sel['t1_dev'])
    train_sc, test_sc = [], []
    for train_idx, test_idx in cv.split(split_sel['X_dev']):
        m_clone = clone(m)
        m_clone.fit(split_sel['X_dev'].iloc[train_idx].values,
                    split_sel['y_dev'].iloc[train_idx].values,
                    sample_weight=split_sel['w_dev'].iloc[train_idx].values)
        train_sc.append(-log_loss(split_sel['y_dev'].iloc[train_idx].values,
                                       m_clone.predict_proba(split_sel['X_dev'].iloc[train_idx].values)))
        test_sc.append(-log_loss(split_sel['y_dev'].iloc[test_idx].values,
                                      m_clone.predict_proba(split_sel['X_dev'].iloc[test_idx].values)))
    
    xgb_results.append({'max_depth': md, 'learning_rate': lr, 'n_estimators': ne,
                        'min_child_weight': mcw,
                        'train_nll': np.mean(train_sc), 'test_nll': np.mean(test_sc),
                        'gap': np.mean(train_sc) - np.mean(test_sc)})

xgb_df = pd.DataFrame(xgb_results).sort_values('test_nll', ascending=False)
print("Top 15 by test neg log-loss:")
print(xgb_df.head(15).to_string(index=False))

Testing 875 XGBoost configurations...

Top 15 by test neg log-loss:
 max_depth  learning_rate  n_estimators  min_child_weight  train_nll  test_nll      gap
         5           0.10           300                15  -0.655653 -0.671370 0.015717
         6           0.10           300                15  -0.655653 -0.671370 0.015717
         4           0.10           300                15  -0.655653 -0.671370 0.015717
         3           0.10           300                15  -0.655653 -0.671370 0.015717
         2           0.10           300                15  -0.655653 -0.671370 0.015717
         5           0.10           200                15  -0.656764 -0.672615 0.015851
         2           0.10           200                15  -0.656764 -0.672615 0.015851
         4           0.10           200                15  -0.656764 -0.672615 0.015851
         3           0.10           200                15  -0.656764 -0.672615 0.015851
         6           0.10           200             

## 13. Pipeline Hyperparameter Tuning

In [23]:
# === CELL: Pipeline Hyperparameter Tuning ===
# Uses best XGBoost params from above. Takes ~15-30 min on your MacBook.

best_model = XGBClassifier(
    max_depth=int(xgb_df.iloc[0]['max_depth']),
    learning_rate=xgb_df.iloc[0]['learning_rate'],
    n_estimators=int(xgb_df.iloc[0]['n_estimators']),
    min_child_weight=int(xgb_df.iloc[0]['min_child_weight']),
    eval_metric='logloss', random_state=42,
)

delta_grid = [1e-6, 1e-5, 1e-4]
num_days_grid = [100, 150, 200]
pt_sl_grid = [1.5, 2.0, 2.5]

pipe_results = []
total = len(delta_grid) * len(num_days_grid) * len(pt_sl_grid)
combo = 0

for delta in delta_grid:
    # Re-run Kalman with this delta
    df_temp = df.drop(columns=[c for c in df.columns if c.startswith('kf_')], errors='ignore')
    df_temp = compute_kalman_hedge(df_temp, 'close_corn', 'close_soybean', delta=delta)
    
    for num_days in num_days_grid:
        for pt_sl_mult in pt_sl_grid:
            combo += 1
            
            # Re-run labeling
            vol_t = get_daily_vol(corn, soy, hedge_ratio)
            t1_t = get_vertical_barrier(valid_dates, num_days=num_days, t_events=valid_dates)
            touches_t = apply_pt_sl_on_t1(corn, soy, hedge_ratio, t1_t, vol_t,
                                          pt_sl=[pt_sl_mult, pt_sl_mult])
            labels_t = get_bins(touches_t, corn, soy, hedge_ratio)
            weights_t = get_avg_uniqueness(labels_t, df.index)
            
            X_t = df_temp.loc[labels_t.index, selected].copy()
            y_t = labels_t['bin'].map({-1: 0, 1: 1})
            t1_t_s, w_t = labels_t['t1'], weights_t
            
            sp = temporal_split(X_t, y_t, t1_t_s, sample_weight=w_t, n_holdout=375)
            
            cv = WalkForwardPurgedCV(n_periods=3, t1=sp['t1_dev'])
            train_sc, test_sc = [], []
            for train_idx, test_idx in cv.split(sp['X_dev']):
                m = clone(best_model)
                m.fit(sp['X_dev'].iloc[train_idx].values, sp['y_dev'].iloc[train_idx].values,
                      sample_weight=sp['w_dev'].iloc[train_idx].values)
                train_sc.append(-log_loss(sp['y_dev'].iloc[train_idx].values,
                                               m.predict_proba(sp['X_dev'].iloc[train_idx].values)))
                test_sc.append(-log_loss(sp['y_dev'].iloc[test_idx].values,
                                              m.predict_proba(sp['X_dev'].iloc[test_idx].values)))
            
            pipe_results.append({
                'delta': delta, 'num_days': num_days, 'pt_sl': pt_sl_mult,
                'n_events': len(labels_t),
                'train_nll': np.mean(train_sc), 'test_nll': np.mean(test_sc),
                'gap': np.mean(train_sc) - np.mean(test_sc),
            })
            
            print(f"[{combo}/{total}] delta={delta:.0e} days={num_days} pt_sl={pt_sl_mult} "
                  f"events={len(labels_t)} train={np.mean(train_sc):.3f} test={np.mean(test_sc):.3f}")

pipe_df = pd.DataFrame(pipe_results).sort_values('test_nll', ascending=False)
print("\nTop 10 pipeline configurations:")
print(pipe_df.head(10).to_string(index=False))

Converged in 4 iteration(s)
  phi      = 0.994075  (half-life = 168 trading days)
  sigma2_s = 0.030082
  delta    = 1.0e-06  (hyperparameter — tune via CV)
  R        = 1.0e-06  (numerical floor)
Hedge ratio:      0.2787 to 0.6499
Spread level std: 1.5955
Innovation z std: 1.1045 (ideal ~ 1.0)
Level z std:      1.1798
Dev set:      2673 obs  (2013-09-20 → 2024-05-03)
Holdout set:   375 obs  (2024-05-06 → 2025-10-31)
Cutoff date: 2024-05-06
Dev label dist:     {0: 1285, 1: 1388}
Holdout label dist: {0: 175, 1: 200}
[1/27] delta=1e-06 days=100 pt_sl=1.5 events=3048 train=-0.542 test=-0.746
Dev set:      2673 obs  (2013-09-20 → 2024-05-03)
Holdout set:   375 obs  (2024-05-06 → 2025-10-31)
Cutoff date: 2024-05-06
Dev label dist:     {0: 1273, 1: 1400}
Holdout label dist: {0: 172, 1: 203}
[2/27] delta=1e-06 days=100 pt_sl=2.0 events=3048 train=-0.632 test=-0.694
Dev set:      2673 obs  (2013-09-20 → 2024-05-03)
Holdout set:   375 obs  (2024-05-06 → 2025-10-31)
Cutoff date: 2024-05-06
Dev l

## 14. Final Holdout Evaluation

Evaluate the final specification on the held-out test set (touched **once**).
Uses best pipeline params from `pipe_df`, best XGBoost params from `xgb_df`,
selected features from cell 18, and Sequential Bootstrap RF.

In [24]:
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix, classification_report

# ── 1. Retrieve best configuration ──────────────────────────────────────────
best_pipe   = pipe_df.iloc[0]
best_delta  = best_pipe['delta']
best_days   = int(best_pipe['num_days'])
best_pt_sl  = best_pipe['pt_sl']

best_xgb    = xgb_df.iloc[0]
final_model = XGBClassifier(
    max_depth=int(best_xgb['max_depth']),
    learning_rate=best_xgb['learning_rate'],
    n_estimators=int(best_xgb['n_estimators']),
    min_child_weight=int(best_xgb['min_child_weight']),
    eval_metric='logloss',
    random_state=42,
)

print(f"Best pipeline:  delta={best_delta:.0e}, num_days={best_days}, pt_sl={best_pt_sl}")
print(f"Best XGBoost:   max_depth={int(best_xgb['max_depth'])}, lr={best_xgb['learning_rate']}, "
      f"n_est={int(best_xgb['n_estimators'])}, mcw={int(best_xgb['min_child_weight'])}")

# ── 2. Reconstruct features + labels with best pipeline params ──────────────
df_final = df.drop(columns=[c for c in df.columns if c.startswith('kf_')], errors='ignore')
df_final = compute_kalman_hedge(df_final, 'close_corn', 'close_soybean', delta=best_delta)

vol_f     = get_daily_vol(corn, soy, hedge_ratio)
t1_f      = get_vertical_barrier(valid_dates, num_days=best_days, t_events=valid_dates)
touches_f = apply_pt_sl_on_t1(corn, soy, hedge_ratio, t1_f, vol_f,
                               pt_sl=[best_pt_sl, best_pt_sl])
labels_f  = get_bins(touches_f, corn, soy, hedge_ratio)
weights_f = get_avg_uniqueness(labels_f, df.index)

X_f  = df_final.loc[labels_f.index, selected].copy()
y_f  = labels_f['bin'].map({-1: 0, 1: 1})
t1_f_s = labels_f['t1']
w_f  = weights_f

sp = temporal_split(X_f, y_f, t1_f_s, sample_weight=w_f, n_holdout=375)

print(f"\nDev set:     {sp['X_dev'].shape[0]} obs  "
      f"({sp['X_dev'].index[0].date()} to {sp['X_dev'].index[-1].date()})")
print(f"Holdout set: {sp['X_holdout'].shape[0]} obs  "
      f"({sp['X_holdout'].index[0].date()} to {sp['X_holdout'].index[-1].date()})")

# ── 3. Train XGBoost on full dev set, evaluate on holdout ────────────────────
final_model.fit(sp['X_dev'].values, sp['y_dev'].values,
                sample_weight=sp['w_dev'].values)

xgb_probs = final_model.predict_proba(sp['X_holdout'].values)
xgb_preds = (xgb_probs[:, 1] >= 0.5).astype(int)
xgb_acc   = accuracy_score(sp['y_holdout'].values, xgb_preds)
xgb_ll    = log_loss(sp['y_holdout'].values, xgb_probs)

print(f"\n{'='*60}")
print(f"  HOLDOUT RESULTS — XGBoost")
print(f"{'='*60}")
print(f"  Accuracy:      {xgb_acc:.4f}")
print(f"  Neg Log-Loss:  {-xgb_ll:.4f}")
print(f"\n  Confusion Matrix:")
cm = confusion_matrix(sp['y_holdout'].values, xgb_preds)
print(f"               Pred 0    Pred 1")
print(f"  Actual 0      {cm[0,0]:>5}     {cm[0,1]:>5}")
print(f"  Actual 1      {cm[1,0]:>5}     {cm[1,1]:>5}")
print(f"\n  Classification Report:")
print(classification_report(sp['y_holdout'].values, xgb_preds,
                            target_names=['Short (0)', 'Long (1)']))

# ── 4. Sequential Bootstrap Random Forest on holdout ─────────────────────────
t1_dev = sp['t1_dev']
avg_u  = sp['w_dev'].mean()
s_length_ho = max(int(avg_u * len(t1_dev)), 50)

print(f"{'='*60}")
print(f"  HOLDOUT RESULTS — Seq Bootstrap RF ({n_trees} trees, bag={s_length_ho})")
print(f"{'='*60}")

rf_trees = []
for tree_i in range(n_trees):
    bag_idx = seq_bootstrap(t1_dev, df.index, s_length=s_length_ho,
                            random_state=tree_i)
    dt = DecisionTreeClassifier(
        criterion='entropy',
        max_features='sqrt',
        max_depth=5,
        min_samples_leaf=max(1, int(0.05 * s_length_ho)),
        class_weight='balanced',
        random_state=tree_i,
    )
    dt.fit(sp['X_dev'].iloc[bag_idx].values, sp['y_dev'].iloc[bag_idx].values,
           sample_weight=sp['w_dev'].iloc[bag_idx].values)
    rf_trees.append(dt)

rf_probs = np.array([t.predict_proba(sp['X_holdout'].values)
                     for t in rf_trees]).mean(axis=0)
rf_preds = (rf_probs[:, 1] >= 0.5).astype(int)
rf_acc   = accuracy_score(sp['y_holdout'].values, rf_preds)
rf_ll    = log_loss(sp['y_holdout'].values, rf_probs)

print(f"  Accuracy:      {rf_acc:.4f}")
print(f"  Neg Log-Loss:  {-rf_ll:.4f}")
print(f"\n  Confusion Matrix:")
cm_rf = confusion_matrix(sp['y_holdout'].values, rf_preds)
print(f"               Pred 0    Pred 1")
print(f"  Actual 0      {cm_rf[0,0]:>5}     {cm_rf[0,1]:>5}")
print(f"  Actual 1      {cm_rf[1,0]:>5}     {cm_rf[1,1]:>5}")
print(f"\n  Classification Report:")
print(classification_report(sp['y_holdout'].values, rf_preds,
                            target_names=['Short (0)', 'Long (1)']))

# ── 5. Summary ───────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"  SUMMARY")
print(f"{'='*60}")
print(f"  {'Model':<25} {'Accuracy':>10} {'Neg Log-Loss':>14}")
print(f"  {'-'*50}")
print(f"  {'XGBoost':<25} {xgb_acc:>10.4f} {-xgb_ll:>14.4f}")
print(f"  {'Seq Bootstrap RF':<25} {rf_acc:>10.4f} {-rf_ll:>14.4f}")
print(f"  {'Baseline (random)':<25} {'0.5000':>10} {-np.log(0.5):>14.4f}")


Best pipeline:  delta=1e-06, num_days=200, pt_sl=2.5
Best XGBoost:   max_depth=5, lr=0.1, n_est=300, mcw=15
Converged in 4 iteration(s)
  phi      = 0.994075  (half-life = 168 trading days)
  sigma2_s = 0.030082
  delta    = 1.0e-06  (hyperparameter — tune via CV)
  R        = 1.0e-06  (numerical floor)
Hedge ratio:      0.2787 to 0.6499
Spread level std: 1.5955
Innovation z std: 1.1045 (ideal ~ 1.0)
Level z std:      1.1798
Dev set:      2573 obs  (2013-09-20 → 2023-12-08)
Holdout set:   375 obs  (2023-12-11 → 2025-06-10)
Cutoff date: 2023-12-11
Dev label dist:     {0: 1205, 1: 1368}
Holdout label dist: {0: 216, 1: 159}

Dev set:     2573 obs  (2013-09-20 to 2023-12-08)
Holdout set: 375 obs  (2023-12-11 to 2025-06-10)

  HOLDOUT RESULTS — XGBoost
  Accuracy:      0.5787
  Neg Log-Loss:  -0.7218

  Confusion Matrix:
               Pred 0    Pred 1
  Actual 0        105       111
  Actual 1         47       112

  Classification Report:
              precision    recall  f1-score   supp